In [24]:
import xarray as xr
import os
import numpy as np
from eofs.standard import Eof

from functions.SIT_functions.SIT_eddy_feedback_functions import eof_calc, vert_integrate, read_daily_averages, propagate_missing_data_to_all_vars

In [25]:
exp_type = 'cmip'
output_eof_file = '/home/users/cturrell/documents/eddy_feedback/b-parameter/cmip6_b/hist_NaN_issues/data'
force_eof_recalculate = True
pfull_slice = slice(850., 100.)
subtract_annual_cycle=True
eof_vars = ['ucomp', 'div1_QG', 'div1_QG_123', 'div1_QG_gt3']
n_eofs = 3
season_month_dict = {'DJF':[12,1,2,]}
propogate_all_nans = True

In [26]:
access_path = '/gws/ssde/j25a/arctic_connect/cturrell/CMIP6/historical/ACCESS-CM2/1950_2015/6hrPlevPt/'
anom_ds_access = xr.open_dataset(os.path.join(access_path, '1979_2015', 'anoms.nc'))
dataset_access = read_daily_averages(os.path.join(access_path, 'yearly_data'), start_month=1979, end_month=2015, exp_type='cmip6')

eof_access = eof_calc('cmip', output_eof_file, force_eof_recalculate, dataset_access, pfull_slice,
                              subtract_annual_cycle, eof_vars, n_eofs, season_month_dict, anom_ds_access,
                              propogate_all_nans, level_subset=[250., 500., 850.],
                              pressure_weighted=True)

2026-04-14 16:35:23,758 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1127) >> reading daily average files
2026-04-14 16:35:23,758 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1127) >> reading daily average files


2026-04-14 16:35:25,055 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1168) >> finished reading daily average files
2026-04-14 16:35:25,055 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1168) >> finished reading daily average files
2026-04-14 16:35:25,082 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1193) >> duplicate times are present!!!
2026-04-14 16:35:25,082 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1193) >> duplicate times are present!!!
2026-04-14 16:35:25,391 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1197) >> duplicate times removed. Was 13185 now 13150
2026-04-14 16:35:25,391 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1197) >> duplicate times removed. Was 13185 now 13150
2026-04-14 16:35:25,415 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1200) >> duplicate times are NOT present - continuing
2026-04-14 16:35:25,415 [INFO]: SIT_eddy_feedback_funct

PermissionError: [Errno 13] Permission denied: '/home/users/cturrell/documents/eddy_feedback/b-parameter/cmip6_b/hist_NaN_issues/data_prop_nans.nc'

In [ ]:


def obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds,
             propagate_all_nans, level_subset=None):  
    
    import logging
    import numpy as np

    pfull_selector = level_subset if level_subset is not None else pfull_slice

    # Resolve level_subset to nearest actual levels in the data to avoid
    # exact-match failures on models with slightly different pressure coordinates
    if isinstance(pfull_selector, list):
        pfull_selector = [float(anom_ds['ucomp_anom'].sel(pfull=lev, method='nearest').pfull.values)
                          for lev in pfull_selector]

    if np.all(dataset.lat.diff('lat')<0.):
        hemisphere_slice_dict = {'n':slice(80.,10.),
                                's':slice(-10., -80.),                          
                                }
    else:
        hemisphere_slice_dict = {'n':slice(10.,80.),
                                's':slice(-80., -10.),                             
                                }        

    if propagate_all_nans:
        output_eof_file_use = output_eof_file.split('.nc')[0]+'_prop_nans.nc'
    else:
        output_eof_file_use = output_eof_file

    if os.path.isfile(output_eof_file_use) and not force_eof_recalculate:
        logging.info('attempting to read in EOF data')
        eof_ds = xr.open_mfdataset(output_eof_file_use, decode_times=True)
        logging.info('SUCCESS')
    else:
        logging.info('failed to read in previously calculated EOF data - CALCULATING')

        logging.info('calculating anomalies')
        if exp_type=='held_suarez':
            u_anoms = dataset['ucomp'].mean('lon') - dataset['ucomp'].mean(('time','lon'))
        else:
            # u_anoms_time = dataset.temporal.departures(data_var = 'ucomp', freq='day', weighted=False)['ucomp'].mean('lon')
            u_anoms_time = anom_ds['ucomp_anom']

        season_list = [season_val for season_val in season_month_dict.keys()]

        all_time_season_list = season_list+['all_time']

        eof_ds = xr.Dataset()
        eof_ds.coords['time'] = ('time', u_anoms_time.time.values)
        eof_ds.coords['pfull'] = ('pfull', u_anoms_time.sel(pfull=pfull_selector).pfull.values)  
        eof_ds.coords['lat'] = ('lat', u_anoms_time.sel(lat=hemisphere_slice_dict['n']).lat.values)                

        for season_val in season_list:
            u_anoms_season = u_anoms_time.where(anom_ds['time'].dt.month.isin(season_month_dict[season_val]), drop=True)
            eof_ds.coords[f'time_{season_val}'] = (f'time_{season_val}', u_anoms_season.time.values)

        eof_ds.coords['eof_num'] = ('eof_num', np.arange(n_eofs))

        ucomp_solver_dict = {}
        ucomp_500_solver_dict = {}        
        ucomp_va_solver_dict = {}        
        for season_val in all_time_season_list:
            ucomp_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_va_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_500_solver_dict[season_val] = {'n':{}, 's':{}}

        logging.info('about to propagate nans to all fields')
        if propagate_all_nans:
            anom_ds = propagate_missing_data_to_all_vars(anom_ds, eof_vars)

        for eof_var in eof_vars:

            logging.info('loading anomalies from anom_ds')

            var_anoms = anom_ds[f'{eof_var}_anom'].load()
            orig_var = anom_ds[f'{eof_var}_orig'].load()           

            for hemisphere in ['n','s']:

                for time_frame in all_time_season_list:
                    var_anoms_hem = var_anoms.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        var_anoms_hem = var_anoms_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True) 
                        time_dim_name = f'time_{time_frame}'
                    else:
                        time_dim_name = 'time'

                    orig_var_hem = orig_var.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        orig_var_hem = orig_var_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True)
                        
                    orig_var_hem = orig_var_hem.mean('time')   
                    
                    va_var_anoms_hem = vert_integrate(var_anoms_hem, pressure_weighted=True)

    return var_anoms_hem, va_var_anoms_hem, anom_ds

orig, va, anom_ds = obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset_access, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds_access, propagate_all_nans=True, level_subset=[250.,500.,850.])
va


2026-04-14 16:13:51,526 [INFO]: 2778376354.py(obtain_orig_var_hem:35) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-14 16:13:51,526 [INFO]: 2778376354.py(obtain_orig_var_hem:35) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-14 16:13:51,528 [INFO]: 2778376354.py(obtain_orig_var_hem:37) >> calculating anomalies
2026-04-14 16:13:51,528 [INFO]: 2778376354.py(obtain_orig_var_hem:37) >> calculating anomalies


2026-04-14 16:13:51,572 [INFO]: 2778376354.py(obtain_orig_var_hem:67) >> about to propagate nans to all fields
2026-04-14 16:13:51,572 [INFO]: 2778376354.py(obtain_orig_var_hem:67) >> about to propagate nans to all fields
2026-04-14 16:13:51,934 [INFO]: SIT_eddy_feedback_functions.py(propagate_missing_data_to_all_vars:660) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 1277263 nans
2026-04-14 16:13:51,934 [INFO]: SIT_eddy_feedback_functions.py(propagate_missing_data_to_all_vars:660) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 1277263 nans
2026-04-14 16:13:51,935 [INFO]: 2778376354.py(obtain_orig_var_hem:73) >> loading anomalies from anom_ds
2026-04-14 16:13:51,935 [INFO]: 2778376354.py(obtain_orig_var_hem:73) >> loading anomalies from anom_ds
2026-04-14 16:13:52,023 [INFO]: 2778376354.py(obtain_orig_var_hem:73) >> loading anomalies from an

<xarray.DataArray (time: 13141, lat: 57)> Size: 6MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])
Coordinates:
  * lat      (lat) float64 456B -80.0 -78.75 -77.5 -76.25 ... -12.5 -11.25 -10.0
  * time     (time) datetime64[ns] 105kB 1979-01-01 1979-01-02 ... 2015-01-01

In [ ]:
print(orig.size)
print(orig.isnull().sum())
print(va.isnull().sum())

print((orig.isnull().sum()/orig.size)*100)

2247111
<xarray.DataArray 'div1_QG_gt3_anom' ()> Size: 8B
array(410986)
<xarray.DataArray ()> Size: 8B
array(410986)
<xarray.DataArray 'div1_QG_gt3_anom' ()> Size: 8B
array(18.2895282)


In [ ]:
def eof_calc_alt(data,lats):

    coslat = np.cos(np.deg2rad(lats)).clip(0., 1.)
    wgts = np.sqrt(coslat)[np.newaxis, np.newaxis, :]

    solver = Eof(data, weights=wgts, center=True)

    eofs = solver.eofsAsCovariance(neofs=3)
    pc1 = solver.pcs(npcs=3, pcscaling=1)

    variance_fractions = solver.varianceFraction(neigs=3)

    return eofs, pc1, variance_fractions, solver

eofs_va, pc1_va, variance_fractions_va, solver_va = eof_calc_alt(va.values, va.lat.values)

In [ ]:
anom_ds.pfull

<xarray.DataArray 'pfull' (pfull: 3)> Size: 24B
array([850., 500., 250.])
Coordinates:
  * pfull    (pfull) float64 24B 850.0 500.0 250.0